In [1]:
## cnn pipeline with raw sequence inputs
## one-hot encoded

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
from tqdm import tqdm


In [4]:
# Mapping for 20 standard amino acids
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

# Directory setup
os.makedirs("cnn_models", exist_ok=True)
os.makedirs("cnn_results", exist_ok=True)



In [5]:
# CNN model
class ProteinCNN(nn.Module):
    def __init__(self, input_dim=20):
        super(ProteinCNN, self).__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x).squeeze(-1)
        x = self.sigmoid(self.fc(x))
        return x.squeeze()

# Dataset class

class ProteinDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

        lengths = [len(seq) for seq in sequences]
        min_len = min(lengths)
        max_len = max(lengths)

        if max_len - min_len > 1:
            raise ValueError(f"Sequences vary too much in length! Found lengths: {set(lengths)}")

        self.seq_len = max_len  # allow minor difference (pad shorter sequences)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]

        onehot = np.zeros((20, self.seq_len), dtype=np.float32)
        for i, aa in enumerate(seq):
            if i >= self.seq_len:  # (safety) but should never happen
                break
            if aa in AA_TO_INDEX:
                onehot[AA_TO_INDEX[aa], i] = 1.0

        return torch.tensor(onehot), torch.tensor(label, dtype=torch.float32)



In [6]:
# Training loop
def train_eval_model(gene_name, df, max_len):
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype"].isin(["R", "S"])]
    sequences = df["Protein_Sequence"].tolist()
    labels = (df["Phenotype"] == "R").astype(int).tolist()

    # dataset = ProteinDataset(sequences, labels, max_len)
    dataset = ProteinDataset(sequences, labels)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)


    model = ProteinCNN()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for epoch in range(10):
        for X, y in loader:
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X, y in loader:
            out = model(X)
            all_preds.extend(out.numpy())
            all_labels.extend(y.numpy())
    auc = roc_auc_score(all_labels, all_preds)

    torch.save(model.state_dict(), f"auc_results/cnn_results/cnn_models/{gene_name}_cnn.pt")

    result_df = pd.DataFrame({
        "Gene": [gene_name],
        "AUC": [auc]
    })
    result_df.to_csv(f"auc_results/cnn_results/{gene_name}_results.csv", index=False)
    return result_df



In [5]:
gene_list=['rpoB','rpsL','tlyA','pncA','eis','gid','katG','inhA','embA','embB', 'embC','gyrB', 'gyrA', 'ethA', 'ethR']

In [ ]:
all_results = []

for gene in tqdm(gene_list):
    print("gene name", gene)
    file_path = f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv"
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        max_len = df["Protein_Sequence"].str.len().max()
        result = train_eval_model(gene, df, max_len)
        all_results.append(result)
print(all_results)
summary_df = pd.concat(all_results, ignore_index=True)
summary_df.to_csv("auc_results/cnn_summary_auc_scores.csv", index=False)


## per residue attribution

In [31]:
# Enable gradient tracking on input.

# Run a forward pass to get model predictions.

# Backpropagate using .backward() on the prediction score (or loss).

# Extract gradients w.r.t. input → these act as importance scores.

# Aggregate across channels (i.e., amino acids) per residue.

# Store per-residue importances across the dataset.

In [7]:
def compute_residue_importance(model, dataloader, device, save_path=None):
    model.eval()
    model.to(device)
    all_importance_scores = []
    all_labels = []
    
    for inputs, labels in tqdm(dataloader, desc="Computing Residue Importance"):
        inputs = inputs.to(device)
        inputs.requires_grad_()
        outputs = model(inputs)

        grads = []
        for i in range(len(inputs)):
            model.zero_grad()
            outputs[i].backward(retain_graph=True)

            grad = inputs.grad[i].detach().cpu().numpy()  # (20, seq_len)
            score = np.abs(grad).sum(axis=0)  # sum across channels
            grads.append(score)

        all_importance_scores.extend(grads)
        all_labels.extend(labels.numpy())

        inputs.requires_grad_(False)
        inputs.grad = None

    importance_matrix = np.array(all_importance_scores)
    labels_array = np.array(all_labels)

    if save_path:
        np.savez(save_path, importance=importance_matrix, labels=labels_array)

    return importance_matrix, labels_array


In [8]:
model = ProteinCNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# #mean 
# for gene in tqdm(gene_list):

#     model.load_state_dict(torch.load(f"auc_results/cnn_results/cnn_models/{gene}_cnn.pt"))
#     # Load the CSV
#     df = pd.read_csv(f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv")

#     # Filter valid samples
#     df = df[df["Frameshift_Mutation"] == 0]
#     df = df[df["Phenotype"].isin(["R", "S"])]
#     sequences = df["Protein_Sequence"].tolist()
#     labels = df["Phenotype"].map({"S": 0, "R": 1}).tolist()

#     # Get max_len per gene
#     max_len = max(len(seq) for seq in sequences)

#     # Create Dataset + DataLoader
#     dataset = ProteinDataset(sequences, labels)
#     dataloader = DataLoader(dataset, batch_size=16, shuffle=False)
    
#     importance_scores, labels = compute_residue_importance(
#         model, dataloader, device,
#         save_path=f"residue_importance/cnn/npz_files/{gene}_cnn_residue_importance.npz"
#     )
#     # Compute mean over sequences (axis=0)
#     global_importance = importance_scores.mean(axis=0)

#     # Save mean score as CSV
#     df = pd.DataFrame({
#         "Residue_Position": list(range(len(global_importance))),
#         "Importance": global_importance
#     })
#     df.to_csv(f"residue_importance/cnn/{gene}_cnn_residue_importance_global.csv", index=False)




### LOO attribution

In [9]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm


# -------------------------
# LOO Attribution
# -------------------------

@torch.no_grad()
def compute_batch_loo_attribution(model, input_tensor, device="cuda"):
    """
    Compute LOO attribution by masking each residue in parallel (batch-efficient).
    
    Args:
        model: Trained CNN or Transformer model.
        input_tensor: torch.Tensor of shape [1, 20, L] (one-hot encoded sequence).
        device: CUDA or CPU device.

    Returns:
        np.array of shape (L,) with LOO importance scores (Δ logits).
    """
    model.eval()
    input_tensor = input_tensor.to(device)  # [1, 20, L]
    input_tensor = input_tensor.clone()

    B, C, L = input_tensor.shape  # B=1, C=20

    # Compute original prediction
    original_output = model(input_tensor).item()

    # Prepare batch of L inputs each with one position masked
    batch_inputs = input_tensor.repeat(L, 1, 1)  # [L, 20, L]
    for i in range(L):
        batch_inputs[i, :, i] = 0.0  # zero out i-th residue

    # Run forward pass
    batch_outputs = model(batch_inputs.to(device))  # shape: [L]
    if batch_outputs.ndim == 0:
        batch_outputs = batch_outputs.unsqueeze(0)
    loo_scores = original_output - batch_outputs.detach().cpu().numpy()

    return loo_scores  # shape: (L,)



In [ ]:

# -------------------------
# Main Loop per Gene
# -------------------------


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gene_list = [ 'rpsL',  'gid', 'katG', 'inhA', 'ethA', 'ethR'] 
for gene in tqdm(gene_list, desc="LOO Attribution"):
    model = ProteinCNN()
    model.load_state_dict(torch.load(f"auc_results/cnn_results/cnn_models/{gene}_cnn.pt"))
    model.to(device)

    df = pd.read_csv(f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv")
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype"].isin(["R", "S"])]
    sequences = df["Protein_Sequence"].tolist()
    labels = df["Phenotype"].map({"S": 0, "R": 1}).tolist()

    max_len = max(len(seq) for seq in sequences)
    dataset = ProteinDataset(sequences, labels)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

    all_loo_scores = []

    for i, (input_tensor, label) in enumerate(tqdm(dataloader, desc=f"{gene} LOO")):
        # loo_score = compute_loo_attribution(model, input_tensor, device)
        loo_score = compute_batch_loo_attribution(model, input_tensor, device)

        all_loo_scores.append(loo_score)

    # Save all LOO scores
    loo_array = np.array(all_loo_scores)  # shape: (N, L)
    np.savez(f"residue_importance/cnn/npz_files/{gene}_cnn_loo_residue_importance.npz",
             importance=loo_array, labels=np.array(labels))

    # Save mean LOO attribution
    mean_loo = loo_array.mean(axis=0)
    pd.DataFrame({
        "Residue_Position": list(range(len(mean_loo))),
        "LOO_Importance": mean_loo
    }).to_csv(f"residue_importance/cnn/{gene}_cnn_residue_importance_loo.csv", index=False)

    print(f"Saved LOO attribution for {gene}")

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

# -------------------------------
# Constants and setup
# -------------------------------
GENE_LIST = ['rpoB', 'rpsL', 'tlyA', 'pncA', 'eis', 'gid', 'katG', 'inhA', 'embA', 'embB', 'embC', 'gyrB', 'gyrA', 'ethA', 'ethR']
K_VALUES = [10]
ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']
IMPORTANCE_DIR = Path("residue_importance/cnn")
CATALOG_PATH = Path("./data/filtered_variants_output.csv")
OUTPUT_PATH = Path("residue_importance/cnn_k10.csv")


In [10]:

# -------------------------------
# Functions
# -------------------------------
def load_catalog(catalog_path, allowed_confidences):
    catalog = pd.read_csv(catalog_path)
    catalog = catalog[
        (catalog["confidence"].isin(allowed_confidences)) &
        (catalog["intersectional"] == True)
    ].copy()
    catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1
    return catalog

def evaluate_gene(gene_name, catalog_df):
    results = []
    
    # ← Use LOO CSV instead of Grad×Input mean
    imp_path = IMPORTANCE_DIR / f"{gene_name}_cnn_residue_importance_loo.csv"
    if not imp_path.exists():
        print(f"Skipping {gene_name}: missing LOO importance file.")
        return []

    # Load LOO scores
    imp_df = pd.read_csv(imp_path)  # expects: Residue_Position, LOO_Importance
    imp_df_sorted = imp_df.sort_values("LOO_Importance", ascending=False)

    # Subset WHO catalog to current gene
    variants_df = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()].copy()
    if variants_df.empty:
        print(f"Skipping {gene_name}: no intersectional variants found.")
        return []

    total_actual_positives = len(np.unique(variants_df["aa_pos_0idx"]))

    for k in K_VALUES:
        topk_df = imp_df_sorted.head(k)
        important_positions = set(topk_df["Residue_Position"])
        true_positions = set(variants_df["aa_pos_0idx"])
        true_positives = len(true_positions.intersection(important_positions))
        total_predictions = len(important_positions)
        
        precision = true_positives / total_predictions if total_predictions > 0 else 0
        recall = true_positives / total_actual_positives if total_actual_positives > 0 else 0
        
        matched_df = variants_df[variants_df["aa_pos_0idx"].isin(important_positions)]
        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()
        identified_variants_str = ", ".join(identified_variants) if identified_variants else "None"

        f1 = (2 * precision * recall) / (precision + recall + 1e-8)

        results.append({
            "gene": gene_name,
            "Total_Resistance_Positions": total_actual_positives,
            "k": k,
            "TP": true_positives,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "identified_variants": identified_variants_str
        })

    return results





In [ ]:
# -------------------------------
# Main script
# -------------------------------

catalog_df = load_catalog(CATALOG_PATH, ALLOWED_CONFIDENCES)
all_summaries = []

for gene_name in GENE_LIST:
    gene_results = evaluate_gene(gene_name, catalog_df)
    all_summaries.extend(gene_results)


In [17]:
final_df = pd.DataFrame(all_summaries)
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved summary to {OUTPUT_PATH}")


Saved summary to residue_importance/cnn_k10.csv


In [ ]:
best_per_gene.to_csv("residue_importance/cnn_pr_overlap_summary.csv", index=False)

## significance testing

In [9]:
# Full implementation: CNN model training + LOO attribution + fold-wise AUC and interpretability

from sklearn.model_selection import StratifiedKFold
import os
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

def train_eval_cnn_kfold(gene_name, df, n_splits=5, max_len=None):
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype"].isin(["R", "S"])]
    sequences = df["Protein_Sequence"].tolist()
    labels = (df["Phenotype"] == "R").astype(int).tolist()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for fold, (train_idx, test_idx) in enumerate(skf.split(sequences, labels), 1):
        print(f"[{gene_name}] Fold {fold}")
        X_train = [sequences[i] for i in train_idx]
        y_train = [labels[i] for i in train_idx]
        X_test = [sequences[i] for i in test_idx]
        y_test = [labels[i] for i in test_idx]

        # Train model
        model = ProteinCNN()
        model.to(device)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        train_loader = DataLoader(ProteinDataset(X_train, y_train), batch_size=32, shuffle=True)

        model.train()
        for epoch in range(10):
            for X, y in train_loader:
                X, y = X.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(X)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()

        # Evaluate
        model.eval()
        test_loader = DataLoader(ProteinDataset(X_test, y_test), batch_size=32)
        preds, truths = [], []
        with torch.no_grad():
            for X, y in test_loader:
                out = model(X.to(device)).cpu()
                preds.extend(out.numpy())
                truths.extend(y.numpy())
        auc = roc_auc_score(truths, preds)

        # Save fold outputs
        fold_dir = f"cnn_results/{gene_name}/fold_{fold}"
        os.makedirs(fold_dir, exist_ok=True)
        torch.save(model.state_dict(), f"{fold_dir}/model.pt")

        # Compute LOO for each test sequence
        loo_all = []
        for X, _ in DataLoader(ProteinDataset(X_test, y_test), batch_size=1):
            score = compute_batch_loo_attribution(model, X.to(device))
            loo_all.append(score)
        loo_array = np.array(loo_all)
        np.save(f"{fold_dir}/loo.npy", loo_array)

        # Save mean LOO attribution
        mean_loo = loo_array.mean(axis=0)
        pd.DataFrame({
            "Residue_Position": list(range(len(mean_loo))),
            "LOO_Importance": mean_loo
        }).to_csv(f"{fold_dir}/loo_importance.csv", index=False)

        # Save AUC
        pd.DataFrame({"Fold": [fold], "AUC": [auc]}).to_csv(f"{fold_dir}/auc.csv", index=False)
        results.append({"Gene": gene_name, "Fold": fold, "AUC": auc})

    # Save all fold-level AUCs
    all_results_df = pd.DataFrame(results)
    all_results_df.to_csv(f"cnn_results/{gene_name}_fold_auc_summary.csv", index=False)
    return all_results_df



In [10]:
for gene in gene_list:
    df = pd.read_csv(f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv")
    train_eval_cnn_kfold(gene, df)

[gyrB] Fold 1
[gyrB] Fold 2
[gyrB] Fold 3
[gyrB] Fold 4
[gyrB] Fold 5
[gyrA] Fold 1
[gyrA] Fold 2
[gyrA] Fold 3
[gyrA] Fold 4
[gyrA] Fold 5
[ethA] Fold 1
[ethA] Fold 2
[ethA] Fold 3
[ethA] Fold 4
[ethA] Fold 5
[ethR] Fold 1
[ethR] Fold 2
[ethR] Fold 3
[ethR] Fold 4
[ethR] Fold 5


## precision recall significance testing -multi gene drugs

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Constants
GENE_SETS = ["embA+embB+embC","ethA+ethR"]
CATALOG_PATH = "./data/filtered_variants_output.csv"
PROTEIN_REF_PATH = "./data/catalog/protein_sequences.csv"
LOO_DIR = "./statistical_test_results/prediction_task/cnn_results"
OUTPUT_DIR = "./statistical_test_results/PR_task/cnn_results"
ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']

# Load WHO catalog and reference protein lengths
catalog_df = load_catalog(CATALOG_PATH, ALLOWED_CONFIDENCES)
ref_df = pd.read_csv(PROTEIN_REF_PATH)
ref_lengths = dict(zip(ref_df["gene"], ref_df["protein_sequence"].str.len()))

def load_mean_loo(gene_set_name, fold):
    path = os.path.join(LOO_DIR, gene_set_name, f"fold_{fold}", "loo.npy")
    return np.load(path).mean(axis=0)

def evaluate_topk_precision_recall_loocnn(gene_name, scores, catalog_df, k_vals, offset):
    gene_catalog = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()]
    if gene_catalog.empty:
        print(f"Skipping {gene_name}: no confirmed WHO variants.")
        return []

    confirmed_positions = set(gene_catalog["aa_pos_0idx"])
    total_actual = len(confirmed_positions)
    print("offset", offset)

    imp_df = pd.DataFrame({
        "Residue_Position": np.arange(len(scores)) + offset, 
        "Importance": scores
    }).sort_values("Importance", ascending=False)
    print(imp_df.head(10))

    rows = []
    for k in k_vals:
        top_k_pos = set(imp_df.head(k)["Residue_Position"])
        true_positives = len(top_k_pos & confirmed_positions)

        precision = true_positives / k
        recall = true_positives / total_actual if total_actual > 0 else 0
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        matched_df = gene_catalog[gene_catalog["aa_pos_0idx"].isin(top_k_pos)]
        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()

        rows.append({
            "gene": gene_name,
            "TP": true_positives,
            "Total_Resistance_Positions": total_actual,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "k": k,
            "identified_variants": ", ".join(identified_variants) if identified_variants else "None"
        })

    return rows

def run_cnn_loocnn_all_genes(gene_sets=GENE_SETS, k_vals=[10]):
    all_rows = []
    for gene_set in gene_sets:
        print(f"\n=== Evaluating {gene_set} ===")
        gene_parts = gene_set.split("+")
        offsets = {}
        offset = 0
        for gene in gene_parts:
            offsets[gene] = offset
            offset += ref_lengths[gene]

        for fold in range(1, 6):
            print(f"[{gene_set}] Fold {fold}")
            scores = load_mean_loo(gene_set, fold)
            for gene in gene_parts:
                start, end = offsets[gene], offsets[gene] + ref_lengths[gene]
                gene_scores = scores[start:end]

                rows = evaluate_topk_precision_recall_loocnn(
                    gene_name=gene,
                    scores=gene_scores,
                    catalog_df=catalog_df,
                    k_vals=k_vals,
                    offset=start
                )
                for r in rows:
                    r.update({"model": "cnn", "fold": fold, "gene_set": gene_set})
                all_rows.extend(rows)
        print(all_rows[:5])

        out_df = pd.DataFrame(all_rows)
        out_df.to_csv(os.path.join(OUTPUT_DIR, f"{gene_set}_cnn_pr_per_fold.csv"), index=False)

    print(" All CNN PR evaluations saved.")
    return all_rows

# Run the full evaluation across all genes
run_cnn_loocnn_all_genes()


## single gene drugs

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Constants
GENES = ["katG", "inhA", "ethA", "ethR", "embB","gid","rpsL"]
CATALOG_PATH = "./data/filtered_variants_output.csv"
PROTEIN_REF_PATH = "./data/catalog/protein_sequences.csv"
LOO_DIR = "./statistical_test_results/prediction_task/cnn_results"
OUTPUT_DIR = "./statistical_test_results/PR_task/cnn_results"
ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']



catalog_df = load_catalog(CATALOG_PATH, ALLOWED_CONFIDENCES)

# Load protein lengths
ref_df = pd.read_csv(PROTEIN_REF_PATH)
ref_lengths = dict(zip(ref_df["gene"], ref_df["protein_sequence"].str.len()))

def load_mean_loo(gene, fold):
    path = os.path.join(LOO_DIR, gene, f"fold_{fold}", "loo.npy")
    return np.load(path).mean(axis=0)

def evaluate_topk_precision_recall_loocnn(gene, scores, k_vals, catalog_df):
    gene_catalog = catalog_df[catalog_df["gene"].str.lower() == gene.lower()]
    if gene_catalog.empty:
        print(f"[{gene}] Skipping: no confirmed WHO variants.")
        return []

    confirmed_positions = set(gene_catalog["aa_pos_0idx"])
    total_actual = len(confirmed_positions)

    imp_df = pd.DataFrame({
        "Residue_Position": np.arange(len(scores)),
        "Importance": scores
    }).sort_values("Importance", ascending=False)
    print(imp_df.head(5))

    rows = []
    for k in k_vals:
        top_k_pos = set(imp_df.head(k)["Residue_Position"])
        true_positives = len(top_k_pos & confirmed_positions)

        precision = true_positives / k
        recall = true_positives / total_actual if total_actual > 0 else 0
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        matched_df = gene_catalog[gene_catalog["aa_pos_0idx"].isin(top_k_pos)]
        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()

        rows.append({
            "gene": gene,
            "TP": true_positives,
            "Total_Resistance_Positions": total_actual,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "k": k,
            "identified_variants": ", ".join(identified_variants) if identified_variants else "None"
        })
        print(rows[:5])

    return rows

def run_single_gene_loocnn(gene_list=GENES, k_vals=[10]):
    all_rows = []
    for gene in gene_list:
        print(f"\n=== Evaluating {gene} ===")
        for fold in range(1, 6):
            print(f"[{gene}] Fold {fold}")
            scores = load_mean_loo(gene, fold)

            rows = evaluate_topk_precision_recall_loocnn(
                gene=gene,
                scores=scores,
                k_vals=k_vals,
                catalog_df=catalog_df
            )

            for r in rows:
                r.update({
                    "model": "cnn",
                    "fold": fold,
                    "gene_set": gene
                })
            all_rows.extend(rows)

        out_df = pd.DataFrame(all_rows)
        out_df.to_csv(os.path.join(OUTPUT_DIR, f"{gene}_cnn_pr_per_fold.csv"), index=False)

    print("All single-gene CNN PR evaluations completed.")
    return all_rows

# Run the evaluation
run_single_gene_loocnn()
